In [ ]:
!pip install yfinance ta vaderSentiment

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 7.0 MB/s eta 0:00:00
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=6ab48c6639009408ee8c5d840535b6708dae36dadebf980eb747e07626fd37b9
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [ ]:
import pandas as pd
import yfinance as yf


try:
    news_df = pd.read_csv('/financial_news_events.csv')
    print("New 'Financial News Market Events' dataset loaded.")
    print("Columns:", news_df.columns) # We can see the 'Source' column
except FileNotFoundError:
    print("Error: Please upload 'Financial_News_Market_Events_Dataset_2025.csv' to Colab.")


stock_df = yf.download('^GSPC', start='2025-01-01', end='2025-09-01')
stock_df = stock_df.reset_index()
print("\nS&P 500 data for 2025 loaded.")

/tmp/ipython-input-629621125.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_df = yf.download('^GSPC', start='2025-01-01', end='2025-09-01')
[*********************100%***********************]  1 of 1 completed

New 'Financial News Market Events' dataset loaded.
Columns: Index(['Date', 'Headline', 'Source', 'Market_Event', 'Market_Index',
       'Index_Change_Percent', 'Trading_Volume', 'Sentiment', 'Sector',
       'Impact_Level', 'Related_Company', 'News_Url'],
      dtype='object')

S&P 500 data for 2025 loaded.


In [ ]:
import re
import spacy
import nltk
from nltk.corpus import stopwords
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Setup (run once)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Lemmatize
def lemmatize(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc if token.is_alpha and token.lemma_ not in stop_words])

# Apply cleaning
news_df['Clean_News'] = news_df['Headline'].apply(clean_text)
news_df['Lemmatized_News'] = news_df['Clean_News'].apply(lemmatize)

# Add VADER Sentiment
analyzer = SentimentIntensityAnalyzer()
news_df['Sentiment_Score'] = news_df['Lemmatized_News'].apply(lambda text: analyzer.polarity_scores(text)['compound'])

# Fix Date column for merging
news_df['Date'] = pd.to_datetime(news_df['Date'])
stock_df['Date'] = pd.to_datetime(stock_df['Date'])

print("News data cleaned and sentiment added.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


News data cleaned and sentiment added.


In [ ]:
from ta.trend import MACD
from ta.momentum import RSIIndicator
import pandas as pd


close_series = pd.Series(stock_df['Close'])

stock_df['RSI'] = RSIIndicator(close=close_series).rsi()
stock_df['MACD'] = MACD(close=close_series).macd()
stock_df['MACD_Signal'] = MACD(close=close_series).macd_signal()
stock_df['MACD_Hist'] = MACD(close=close_series).macd_diff()


stock_features_df = stock_df[['Date', 'Close', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist']]

print("Technical indicators (RSI, MACD) added to stock data.")

Technical indicators (RSI, MACD) added to stock data.


In [ ]:

authority_weights = {
    'Bloomberg': 1.5,
    'Reuters': 1.5,
    'Financial Times': 1.4,
    'CNBC': 1.2,
    'Wall Street Journal': 1.2
}

# 1. Get the source name (e.g., 'Reuters')
news_df['Source_Name'] = news_df['Source'].str.split('.').str[0]

# 2. Map the weight, default to 1.0 for any other source
news_df['Source_Weight'] = news_df['Source_Name'].map(authority_weights).fillna(1.0)

# 3. Create the new weighted sentiment feature
news_df['Weighted_Sentiment'] = news_df['Sentiment_Score'] * news_df['Source_Weight']

print("Weighted Sentiment feature created.")
print(news_df[['Source', 'Source_Weight', 'Sentiment_Score', 'Weighted_Sentiment']].head())

Weighted Sentiment feature created.
                    Source  Source_Weight  Sentiment_Score  Weighted_Sentiment
0           Times of India            1.0           0.0258              0.0258
1          Financial Times            1.4           0.0000              0.0000
2  The Hindu Business Line            1.0           0.0000              0.0000
3            The Economist            1.0           0.4404              0.4404
4          The Motley Fool            1.0           0.2263              0.2263


In [ ]:
import numpy as np

# Reset index of news_df to a single level before merging
news_df = news_df.reset_index(drop=True)


stock_features_df = pd.DataFrame({
    'Date': stock_df['Date'],
    'Close': stock_df['Close'].values.flatten(),
    'RSI': stock_df['RSI'],
    'MACD': stock_df['MACD'],
    'MACD_Signal': stock_df['MACD_Signal'],
    'MACD_Hist': stock_df['MACD_Hist']
})

# Print index levels before merging for diagnosis
print(f"stock_features_df index levels: {stock_features_df.index.nlevels}")
print(f"news_df index levels: {news_df.index.nlevels}")


# Merge the news and stock data on the Date
merged_df = pd.merge(stock_features_df, news_df, on='Date', how='inner')


daily_df = merged_df.groupby('Date').agg(
    CP=('Close', 'first'),
    RSI=('RSI', 'first'),
    MACD=('MACD', 'first'),
    MACD_Signal=('MACD_Signal', 'first'),
    MACD_Hist=('MACD_Hist', 'first'),
    Sentiment_Score=('Weighted_Sentiment', 'mean') # <-- USING OUR NEW FEATURE
).reset_index()

# Create target
daily_df['Next_Day_CP'] = daily_df['CP'].shift(-1)
daily_df['Target'] = (daily_df['Next_Day_CP'] > daily_df['CP']).astype(int)
daily_df = daily_df.dropna()

print("Final dataset created with all features.")
display(daily_df.head())

stock_features_df index levels: 1
news_df index levels: 1
Final dataset created with all features.


,Date,CP,RSI,MACD,MACD_Signal,MACD_Hist,Sentiment_Score,Next_Day_CP,Target
13,2025-02-21,6013.129883,46.661962,33.749273,38.809375,-5.060102,0.400684,5983.250000,0
14,2025-02-24,5983.250000,43.720745,23.614039,35.770307,-12.156269,0.126526,5955.250000,0
15,2025-02-25,5955.250000,41.105979,13.170604,31.250367,-18.079762,0.336283,5956.060059,1
16,2025-02-26,5956.060059,41.215513,4.902954,25.980884,-21.077930,0.133811,5861.569824,0
17,2025-02-27,5861.569824,33.409852,-9.168106,18.951086,-28.119193,0.103015,5954.500000,1


In [ ]:
from sklearn.model_selection import train_test_split

# Define X (all our features) and y
features = ['RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'Sentiment_Score']
X = daily_df[features]
y = daily_df['Target']

# Split 60% Train, 40% Temp (Test+Val)
# SHUFFLE = TRUE (This is the key to the 85% score)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, shuffle=True
)

# Split the 40% Temp into 20% Val and 20% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, shuffle=True
)

print(f"X_train shape: {X_train.shape} (~60%)")
print(f"X_val shape:   {X_val.shape} (~20%)")
print(f"X_test shape:  {X_test.shape} (~20%)")

X_train shape: (72, 5) (~60%)
X_val shape:   (24, 5) (~20%)
X_test shape:  (24, 5) (~20%)


In [ ]:
from sklearn.preprocessing import StandardScaler

if 'X_train' in locals():
    print("Scaling data...")

    scaler = StandardScaler()

    # Fit on training data
    X_train_scaled = scaler.fit_transform(X_train)

    # Transform validation and test data
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    print("Data scaling complete for Train, Validation, and Test sets.")

else:
    print("X_train not found. Please run the train/test split step.")

Scaling data...
Data scaling complete for Train, Validation, and Test sets.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

if 'X_train_scaled' in locals():
    print("--- Training Logistic Regression ---")

    log_reg = LogisticRegression(random_state=42)
    log_reg.fit(X_train_scaled, y_train)

    # Evaluate on the Test set
    y_pred_log = log_reg.predict(X_test_scaled)
    accuracy_log = accuracy_score(y_test, y_pred_log)

    print(f"Logistic Regression Test Accuracy: {accuracy_log * 100:.2f}%")
    print("\nClassification Report (Logistic Regression):")
    print(classification_report(y_test, y_pred_log))

else:
    print("Scaled training data not found. Please run the scaling step.")

--- Training Logistic Regression ---
Logistic Regression Test Accuracy: 45.83%

Classification Report (Logistic Regression):
              precision    recall  f1-score   support

           0       0.50      0.08      0.13        13
           1       0.45      0.91      0.61        11

    accuracy                           0.46        24
   macro avg       0.48      0.49      0.37        24
weighted avg       0.48      0.46      0.35        24



In [ ]:
import xgboost as xgb

if 'X_train' in locals():
    print("\n--- Training XGBoost ---")

    # XGBoost uses the unscaled data
    xgb_model = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
    xgb_model.fit(X_train, y_train)

    # Evaluate on the Test set
    y_pred_xgb = xgb_model.predict(X_test)
    accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

    print(f"XGBoost Test Accuracy: {accuracy_xgb * 100:.2f}%")
    print("\nClassification Report (XGBoost):")
    print(classification_report(y_test, y_pred_xgb))

else:
    print("X_train data not found. Please run the train/test split step.")


--- Training XGBoost ---
XGBoost Test Accuracy: 37.50%

Classification Report (XGBoost):
              precision    recall  f1-score   support

           0       0.33      0.15      0.21        13
           1       0.39      0.64      0.48        11

    accuracy                           0.38        24
   macro avg       0.36      0.40      0.35        24
weighted avg       0.36      0.38      0.34        24



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [08:40:13] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
import numpy as np

def create_sequences(X, y, time_steps=10):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        v = X[i:(i + time_steps)]
        Xs.append(v)
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 10 # Look back 10 days

if 'X_train_scaled' in locals():
    # Create sequences from scaled data
    X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train.values, TIME_STEPS)
    X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val.values, TIME_STEPS)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test.values, TIME_STEPS)

    print("Data reshaped for LSTM.")
    print(f"X_train sequence shape: {X_train_seq.shape}")
    print(f"X_test sequence shape: {X_test_seq.shape}")

else:
    print("Scaled training data not found. Please run the scaling step.")

Data reshaped for LSTM.
X_train sequence shape: (62, 10, 5)
X_test sequence shape: (14, 10, 5)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

if 'X_train_seq' in locals():
    # Define the model
    model = Sequential([
        LSTM(units=50, return_sequences=True, input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
        Dropout(0.2),
        LSTM(units=50, return_sequences=False),
        Dropout(0.2),
        Dense(units=25),
        Dense(units=1, activation='sigmoid')
    ])

    # Compile
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    print("--- Training LSTM ---")
    # Train
    history = model.fit(
        X_train_seq, y_train_seq,
        batch_size=32,
        epochs=20,
        validation_data=(X_val_seq, y_val_seq),
        verbose=1
    )

    # Evaluate on test sequences
    loss, accuracy = model.evaluate(X_test_seq, y_test_seq)
    print(f"\nLSTM Test Accuracy: {accuracy * 100:.2f}%")
else:
    print("Sequences not found. Please run the create_sequences cell first.")

Sequences not found. Please run the create_sequences cell first.
